# Stringybark Creek — optimisation from Python

Companion notebook for **Tutorial 13 — Optimisation from Python**. It calibrates the Stringybark model with `kalix.optimise()`, inspects the result, then simulates and plots the calibrated model. Run the cells top to bottom.

## 1. Install and import

`pip install kalix` if you haven't already — the optimiser runs in-process, no separate CLI needed.

In [ ]:
import kalix
import pandas as pd

print(f"kalix version: {kalix.__version__}")

## 2. Run the optimisation

`kalix.optimise()` takes the optimisation config, the model to calibrate (`model_file`), and where to write the result (`save_model`). It returns a results dictionary. By default it shows a lightweight progress line while it runs — in a notebook it updates a single line in place — and the run takes about 30 s here.

To handle progress yourself, pass `progress=<callable>`; it's called once per generation with a dict (`n_evaluations`, `best_objective`, `elapsed_seconds`), and the built-in line is suppressed. Pass `progress=False` for no output.

In [ ]:
result = kalix.optimise(
    "optimisation_config.ini",
    model_file="stringybark.ini",
    save_model="stringybark_calibrated.ini",
)

print(f"Done!")

## 3. Inspect the result

The returned dict carries the run summary and the optimised parameters.

In [ ]:
print(f"success:        {result['success']}")
print(f"message:        {result['message']}")
print(f"evaluations:    {result['n_evaluations']}")
print(f"best objective: {result['best_objective']:.3f}")

The `parameters` entry is a `{target: physical_value}` dict — drop it straight into pandas:

In [ ]:
params = pd.Series(result["parameters"], name="value").to_frame()
params

## 4. Simulate the calibrated model

`save_model` already wrote `stringybark_calibrated.ini` to disk. It's an ordinary model file, so simulate it exactly as in Tutorial 5.

In [ ]:
kalix.simulate("stringybark_calibrated.ini", output_file="calibrated_results.csv")

In [ ]:
sim = pd.read_csv("calibrated_results.csv", parse_dates=[0], index_col=0)
obs = pd.read_csv("../data/observed.csv", parse_dates=["Date"], index_col="Date")
sim.head()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
window = slice("1989-01-01", "1990-12-31")
sim.loc[window, "node.0001_sc_stringybark.ds_1"].plot(ax=ax, label="calibrated", linewidth=1.2)
obs.loc[window, "obs"].plot(ax=ax, label="observed", alpha=0.7, linewidth=1.0)
ax.set_ylabel("Flow (ML/day)")
ax.set_title("Stringybark Creek — calibrated vs observed (1989\u20131990)")
ax.legend();

## 5. (bonus) The optimised model as a string

`optimised_model_ini` is the calibrated model serialised back to INI text — the same content `save_model` wrote to disk, handy if you'd rather keep it in memory or post-process it.

In [ ]:
print(result["optimised_model_ini"])

## What to try next

- Loop a scale factor over the rainfall and re-optimise each scenario, stacking the resulting parameter sets into one DataFrame.
- Swap `statistic = SDEB` in the config for `ONE_MINUS_LNSE` and compare the calibrated low-flow fit.
- Add `random_seed = 42` to the config's `[optimisation]` section for reproducible runs.